# Start of Notebook

In [ ]:
import os
import warnings
import random
import pickle
from datetime import datetime
from collections import defaultdict
from functools import partial

# Environment configuration
os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=8'

import numpyro
from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS, log_likelihood, hmc
from numpyro.infer.util import initialize_model
from numpyro.util import fori_collect

import jax
from jax import config
config.update("jax_enable_x64", True)
config.update('jax_platform_name', 'cpu')

import jax.numpy as jnp
from jax import jit, pmap, devices, device_get, lax, local_device_count, random, vmap, block_until_ready
from jax.random import PRNGKey, split
from jax.scipy.optimize import minimize
from jax.scipy.signal import fftconvolve
from jax.scipy.signal import convolve as jax_convolve

# Check devices
print('Available devices:', devices())
print('CPU devices:', devices('cpu'))

# Suppress warnings
warnings.filterwarnings('ignore', message="It appears that you're using a Mac with one of Apple's ARM-based processors")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, to_hex
import corner

import astropy.units as u
import astropy.constants as c
from astropy.constants import G, m_p
from astropy.table import Table

import shone

import fleck
from fleck.jax import ActiveStar

from scipy.optimize import fmin_powell, curve_fit
from scipy.stats import gaussian_kde

from chromatic import *
from svo_filters import svo
from bt_settl import get_interp_stellar_spectrum, get_interp_stellar_spectrum_3d
# from bt_settl_3d import get_interp_stellar_spectrum_3d

import arviz
import arviz as az

from tqdm.auto import tqdm
from glob import glob

# Create custom binned bin edges
wl_min, wl_max = 0.7, 1.7
n_points = 10000

binned_bin_edges = np.geomspace(wl_min, wl_max, n_points + 1)

binned_wavelengths = (binned_bin_edges[:-1] + binned_bin_edges[1:]) / 2.0

print(f"Creating custom binned grid with {n_points} points from {wl_min} to {wl_max} µm")
print(f"Binned bin edges shape: {len(binned_bin_edges)}")
print(f"Binned wavelengths shape: {len(binned_wavelengths)}")

# Step 2: Create the interpolation grid using these custom bin edges
print("\nCreating BT-Settl 3D interpolation grid with custom binning...")
panchromatic_btsettl_grid = get_interp_stellar_spectrum_3d(binned_bin_edges)
panchromatic_wavelengths = binned_wavelengths
panchromatic_bin_edges = binned_bin_edges

print(f"\nWavelength range: {panchromatic_wavelengths[0]:.3f} - {panchromatic_wavelengths[-1]:.3f} µm")
print(f"Number of wavelength points: {len(panchromatic_wavelengths)}")
print("\nGrid creation complete!")

In [ ]:
visits = {
    'F21': {
        'Grism': 'G141',
        # 'Forward': G_141_for_dict,
        # 'Backward': G_141_back_dict,
        'BJD_times': np.array(pd.read_csv('../../data/F21_bjdtimes.csv')['BJD'][:]) * u.day,
        'time_lower': 2459455.708 * u.day,
        'time_upper': 2459455.738 * u.day,
        'T0 (BJD_TDB)': 2459455.9895 * u.day,
        'exp (s)': 4.9784 * u.s,
        'native resolution': 46.3 * u.angstrom
    },
    'S22': {
        'Grism': 'G102',
        # 'Forward': G_102_for_dict,
        # 'Backward': G_102_back_dict,
        'BJD_times': np.array(pd.read_csv('../../data/S22_bjdtimes.csv')['BJD'][:]) * u.day,
        'time_lower': 2459684.215 * u.day,
        'time_upper': 2459684.243 * u.day,
        'T0 (BJD_TDB)': 2459684.4959 * u.day, # This is 27 planetary orbits after the first transit, + 0.0054 days (the transit arrived 7 minutes late)
        'exp (s)': 9.67632 * u.s,
        'native resolution': 24.6 * u.angstrom
    }
}

systeminfo = {
    'duration (hr)': 3.5 * u.hr,
    'T_orb (d)': 8.463 * u.day,
    'T_rot (d)': 4.86 * u.day,
    'inclination': 89.5,
    'eccentricity': 0.0,
    'longitude_of_periastron': 88.4
}

def read_sensitivity_curve(grism='G141'):
    path = f'../../data/WFC3.IR.{grism}.1st.sens.2.fits'

    response = fits.open(path)

    w = response[1].data['wavelength']/1e4 * u.micron
    s = response[1].data['sensitivity'] * u.cm * u.cm / u.erg
    e = response[1].data['error'] * u.cm * u.cm / u.erg
    
    return w, s, e

@jit
def get_planck_spectrum_jax(T, **kwargs):
    """
    Calculate the surface flux from a thermally emitted surface,
    according to Planck function.

    Parameters
    ----------
    wavelength : Quantity
        The wavelengths at which to calculate,
        with units of wavelength.
    temperature : Quantity
        The temperature of the thermal emitter,
        with units of K.

    Returns
    -------
    surface_flux : Quantity
        The surface flux, evaluated at the wavelengths.
    """

    # define variables as shortcut to the constants we need
    h = 6.62607e-27 # erg s
    k = 1.380649e-16 # erg/K
    c = 2.9979e18 # angstrom/s
    wavelength = panchromatic_wavelengths*1e4

    z = h * c / (wavelength * k * T) # units check out

    # calculate the intensity from the Planck function
    intensity = (2 * h * c**2 / wavelength**5 / (jnp.exp(z) - 1)) # Units are erg/s/A^3

    # calculate the flux assuming isotropic emission
    flux = jnp.pi * intensity * 1e16 # erg / (s * cm^2 * angstrom)

    # return the intensity
    wave_jax = jnp.array(panchromatic_wavelengths)
    flux_jax = jnp.array(flux)

    return wave_jax, flux_jax

@jit
def convolve_spectrum_jax(model_wavelength, model_flux, sigma, kernel_size=7, **kwargs):
    """
    Properly convolve a spectrum with a Gaussian kernel in JAX.
    
    Args:
        model_wavelength: Array of wavelengths (must be evenly spaced!)
        model_flux: Corresponding flux values
        sigma: Standard deviation of Gaussian kernel in wavelength units
        kernel_size: Number of elements in the kernel (odd number recommended)
        
    Returns:
        Convolved flux array
    """
    # Ensure inputs are JAX arrays
    model_wavelength = jnp.asarray(model_wavelength)
    model_flux = jnp.asarray(model_flux)
    
    # Create proper Gaussian kernel
    x = jnp.linspace(-(kernel_size//2), kernel_size//2, kernel_size)
    kernel = jnp.exp(-0.5 * (x/sigma)**2)
    kernel = kernel / jnp.sum(kernel)  # normalize
    
    # Perform convolution
    convolved = jax_convolve(model_flux, kernel, mode='same', method='fft')
    
    return convolved

@jit
def get_BTSettl_spectrum_jax(T, 
                             metallicity, 
                             grid=panchromatic_btsettl_grid, **kwargs):
    """
    Get BT-Settl spectrum for given temperature, logg, and metallicity.
    
    Parameters:
    T : effective temperature (K)
    logg : surface gravity (log10(cm/s^2))
    metallicity : metallicity ([M/H])
    grid : 3D interpolation function from get_interp_stellar_spectrum_3d
    
    Returns:
    wave_jax : wavelength array (micrometers)
    flux_jax : flux array (erg/cm^2/s/Å)
    """
    
    # Pass all three parameters to the 3D grid interpolation
    gridspec = grid(
        jnp.array(T, dtype=jnp.float64),
        # jnp.array(logg, dtype=jnp.float32),
        jnp.array(metallicity, dtype=jnp.float64)
    )
    
    # sigma_sb = 5.67e-5  # erg/cm^2/s/K^4
    # nf = (sigma_sb * (T)**4) / (jnp.trapezoid(gridspec, x=jnp.array(panchromatic_wavelengths)*1e4))
    re_normed_flux = gridspec #* nf
    
    wave_jax = jnp.array(panchromatic_wavelengths)
    flux_jax = jnp.array(re_normed_flux)
    
    return wave_jax, flux_jax

## Load Data

In [ ]:
###############################
# S22
###############################

visit = 'S22'
'Things that automatically get re-defined for either visit'
predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
binwidth = visits[f'{visit}']['native resolution']
exptime = visits[f'{visit}']['exp (s)']
grism = visits[f'{visit}']['Grism']
rainbow = read_rainbow(f"../../data/rainbows/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")
speclc_rainbow = rainbow
'remove the transit'
out_of_transit = ( speclc_rainbow.trim().mask_transit(period=8.463*u.day, t0=predicted_T0*u.day, duration=0.15*u.day) )
# out_of_transit.imshow()
'define the appropriate arrays'
_meanOOTspec = out_of_transit.get_average_spectrum_as_rainbow()
meanOOTspec = _meanOOTspec.flux
meanOOTspec_relative_err = _meanOOTspec.uncertainty.value/meanOOTspec.value
oot_spec_err = meanOOTspec_relative_err.flatten()
_SED_wavelengths = (_meanOOTspec.wavelength.value).flatten()
e_per_s = meanOOTspec / exptime
e_per_s_per_angstrom = e_per_s / binwidth
_w, _s, _e = read_sensitivity_curve(grism=grism)
binned_filter_response = bintogrid(_w.value, _s.value, newx=_SED_wavelengths)['y'] * u.cm**2 / u.erg
_calibrated_mean_spec = e_per_s_per_angstrom.flatten() / binned_filter_response
calibrated_mean_spec = _calibrated_mean_spec.value/np.nanmean(_calibrated_mean_spec.value)
oot_spec_err = oot_spec_err * calibrated_mean_spec
# Convert to JAX arrays
calibrated_mean_spec_G102 = jnp.array(calibrated_mean_spec)
oot_spec_err_G102 = jnp.array(oot_spec_err)
SED_wavelengths_G102 = jnp.array(_SED_wavelengths)

###############################
# F21
###############################
visit = 'F21'
'Things that automatically get re-defined for either visit'
predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
binwidth = visits[f'{visit}']['native resolution']
exptime = visits[f'{visit}']['exp (s)']
grism = visits[f'{visit}']['Grism']
rainbow = read_rainbow(f"../../data/rainbows/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")
speclc_rainbow = rainbow
'remove the transit'
out_of_transit = ( speclc_rainbow.trim().mask_transit(period=8.463*u.day, t0=predicted_T0*u.day, duration=0.15*u.day) )
'define the appropriate arrays'
_meanOOTspec = out_of_transit.get_average_spectrum_as_rainbow()
meanOOTspec = _meanOOTspec.flux
meanOOTspec_relative_err = _meanOOTspec.uncertainty.value/meanOOTspec.value
_oot_spec_err = meanOOTspec_relative_err.flatten()
_SED_wavelengths = (_meanOOTspec.wavelength.value).flatten()
e_per_s = meanOOTspec / exptime
e_per_s_per_angstrom = e_per_s / binwidth
_w, _s, _e = read_sensitivity_curve(grism=grism)
binned_filter_response = bintogrid(_w.value, _s.value, newx=_SED_wavelengths)['y'] * u.cm**2 / u.erg
_calibrated_mean_spec = e_per_s_per_angstrom.flatten() / binned_filter_response
calibrated_mean_spec = _calibrated_mean_spec.value/np.nanmean(_calibrated_mean_spec.value)
oot_spec_err = _oot_spec_err * calibrated_mean_spec
# Convert to JAX arrays
calibrated_mean_spec_G141 = jnp.array(calibrated_mean_spec)
oot_spec_err_G141 = jnp.array(oot_spec_err)
SED_wavelengths_G141 = jnp.array(_SED_wavelengths)

###############################
# PLOT DATA
###############################
plt.figure()
plt.errorbar(SED_wavelengths_G102,calibrated_mean_spec_G102,yerr=oot_spec_err_G102*100,fmt='o',label='G102',ms=1)
plt.errorbar(SED_wavelengths_G141,calibrated_mean_spec_G141,yerr=oot_spec_err_G141*100,fmt='o',label='G141',ms=1)
plt.legend()
plt.show()
plt.clf()

# Define Numpyro Model

In [ ]:
def numpyro_model(N_temps=2, return_max_likelihood=False):
    """
    Define the probabilistic model in NUMPYRO with N_temps components.
    
    Args:
        N_temps: Number of temperature components (1, 2, 3, or 4)
        return_max_likelihood: If True, also return the maximum log likelihood value
    """
    def model():
        temp_bounds = [
            (3600, 4200),   # T1 bounds (higher temperature)
            (1000, 3600),   # T2 bounds (cooler component)
            (4500, 10000),  # T3 bounds 
        ]
        
        'Spectral Decomp Params'
        T_samples = []
        for i in range(1, N_temps + 1):
            if i == 1:
                T = numpyro.sample(f'T{i}', dist.TruncatedNormal(loc=4000, scale=50,
                                                              low=3700, high=4300))  
            if i == 2:
                T = numpyro.sample(f'T{i}', dist.TruncatedNormal(loc=3000, scale=71,
                                                              low=2600, high=3600))      
            if i ==3:
                T = numpyro.sample(f'T{i}', dist.Uniform(*temp_bounds[i-1]))
            T_samples.append(T)
        ff_samples = [1.0]  # First component always has f=1
        for i in range(2, N_temps + 1):
            ff = 10**numpyro.sample(f'log_ff{i}', dist.Uniform(-2, 1))
            ff_samples.append(ff)
        'Gridpoint Params'
        metallicity = 0.0
        sigma_conv = 1.0 #numpyro.sample('sigma_conv', dist.Uniform(0.3, 4.0))
        
        # wl_shift_G102 = numpyro.sample('wl_shift_G102', dist.Uniform(-0.001, 0))
        # wl_shift_G141 = numpyro.sample('wl_shift_G141', dist.Uniform(0, 0.005))
        # wl_stretch_G102 = numpyro.sample('wl_stretch_G102', dist.Uniform(1.0, 1.01))
        # wl_stretch_G141 = numpyro.sample('wl_stretch_G141', dist.Uniform(0.98, 1.03))

        'Data Params'
        offset_G102 = numpyro.sample('offset_G102', dist.Uniform(0.0, 1.0))
        slope_G141 = numpyro.sample('slope_G141', dist.Uniform(-1.5, 0.5))
        slope_G102 = numpyro.sample('slope_G102', dist.Uniform(-1.5, 0.5))
        beta_102 = numpyro.sample('beta_G102', dist.Uniform(2.0, 2.5))
        beta_141 = numpyro.sample('beta_G141', dist.Uniform(2.0, 2.6))

        'Apply data params'
        offset_102 = jnp.asarray(offset_G102, dtype=jnp.float64)
        spec_G102_with_offset = jnp.array(calibrated_mean_spec_G102 + offset_102)
        spec_G102_with_slope = spec_G102_with_offset + slope_G102 * (SED_wavelengths_G102 - SED_wavelengths_G102[0])
        spec_G141_with_slope = jnp.array(calibrated_mean_spec_G141) + slope_G141 * (SED_wavelengths_G141 - SED_wavelengths_G141[0])
        _spec_flux_jax = jnp.concatenate([spec_G102_with_slope, spec_G141_with_slope])
        
        spec_flux_jax = _spec_flux_jax / jnp.mean(_spec_flux_jax)
        spec_err_jax = jnp.concatenate(
            [jnp.array((10**beta_102) * oot_spec_err_G102),
             jnp.array((10**beta_141) * oot_spec_err_G141)])
        
        # NEW: Adjust the observed wavelengths for each grism separately
        # adjusted_wavelengths_G102 = (SED_wavelengths_G102 - jnp.median(SED_wavelengths_G102)) * wl_stretch_G102 + jnp.median(SED_wavelengths_G102) + wl_shift_G102
        # adjusted_wavelengths_G141 = (SED_wavelengths_G141 - jnp.median(SED_wavelengths_G141)) * wl_stretch_G141 + jnp.median(SED_wavelengths_G141) + wl_shift_G141
        # combined_SED_wavelengths = jnp.concatenate([adjusted_wavelengths_G102, adjusted_wavelengths_G141])
        combined_SED_wavelengths = jnp.concatenate([SED_wavelengths_G102, SED_wavelengths_G141])

        # combined_SED_wavelengths = jnp.delete(combined_SED_wavelengths, 239, assume_unique_indices=True)
        # spec_flux_jax = jnp.delete(spec_flux_jax, 239, assume_unique_indices=True)
        # spec_err_jax = jnp.delete(spec_err_jax, 239, assume_unique_indices=True)
        # combined_SED_wavelengths = jnp.delete(combined_SED_wavelengths, -5, assume_unique_indices=True)
        # spec_flux_jax = jnp.delete(spec_flux_jax, -5, assume_unique_indices=True)
        # spec_err_jax = jnp.delete(spec_err_jax, -5, assume_unique_indices=True)
        @jit
        def decomp_model(temps, scales, metal_val, sigma):
            # Get spectra for all temperatures with variable metallicity AND logg
            spectra = [get_BTSettl_spectrum_jax(T=Temp, metallicity=metal_val) 
                        for Temp in temps]
            
            # Calculate combined spectrum with scaling factors
            flux_combined = scales[0] * spectra[0][1]  # Start with first component
            for i in range(1, N_temps):
                flux_combined += scales[i] * spectra[i][1]
            
            # Normalize
            _model_flux = flux_combined / jnp.mean(flux_combined)
            
            # NEW: Bin the model to the ADJUSTED wavelength grid
            _binned_model_flux = shone.bin_spectrum(
                combined_SED_wavelengths,  # Target is the adjusted grid
                panchromatic_wavelengths,   # Source is the high-res grid
                _model_flux
            )
            # Convolve and bin (convolution happens on the adjusted grid)
            binned_model_flux = convolve_spectrum_jax(
                combined_SED_wavelengths,   # Convolve on the adjusted grid
                _binned_model_flux, 
                sigma=jnp.asarray(sigma, dtype=jnp.float64)
            )

            model_flux = binned_model_flux / jnp.mean(binned_model_flux)
            
            # Return full model, but we'll slice off the edges where binning may be bad
            return model_flux[2:-7]

        # Calculate model flux with both metallicity AND logg
        model_flux = decomp_model(T_samples, ff_samples, metallicity, sigma_conv)
        
        # Define the likelihood
        log_likelihood = dist.Normal(model_flux, spec_err_jax[2:-7]).log_prob(spec_flux_jax[2:-7]).sum()
        
        # Sample the observed data
        numpyro.sample("Obs", dist.Normal(model_flux, spec_err_jax[2:-7]), obs=spec_flux_jax[2:-7])
        
        # Track log likelihood for maximum likelihood calculation
        if return_max_likelihood:
            numpyro.deterministic('log_likelihood', log_likelihood)
    
    if return_max_likelihood:
        def get_max_likelihood(samples):
            """Extract maximum log likelihood from samples"""
            log_lik_samples = samples.get('log_likelihood', None)
            if log_lik_samples is not None:
                return float(jnp.max(log_lik_samples))
            return None
        
        return model, get_max_likelihood
    else:
        return model

## Define Checkpoint Function

In [ ]:
rng_seed = 0

def hstack_recursive(final_states, checkpoint_states):
    for key in final_states.keys():
        if isinstance(final_states[key], dict):
            hstack_recursive(final_states[key], checkpoint_states[key])
        else:
            final_states[key] = jnp.hstack([
                final_states[key], 
                checkpoint_states[key]
            ])

def print_big_message(big_message):
    print('\n\n')
    print('=' * len(big_message))
    print(big_message)
    print('=' * len(big_message))
    print('\n\n')

def post_batch_viz_save(self, **kwargs):
    """
    here we define some tasks to do after each completed checkpoint:
    """
    print(f'Corner for checkpoint {self.checkpoint}')

    samples_cumulative = self.get_samples()
    corner.corner(samples_cumulative)
    plt.suptitle(f'checkpoint {self.checkpoint}')
    plt.savefig(f'../../figs/joint-visit_chkpt_{self.checkpoint}_sed_corner.png', dpi=200)
    plt.show()
    plt.clf()
    
    # EXTRACT MEDIAN PARAMETERS DIRECTLY FROM SAMPLES (like corner plot does)
    chkpt_median_params = {}
    for param_name, param_samples in samples_cumulative.items():
        # param_samples has shape (num_chains, num_samples)
        # Flatten across chains and compute median
        flattened_samples = param_samples.flatten()
        chkpt_median_params[param_name] = float(jnp.median(flattened_samples))
    
    print("Median parameters from samples:")
    for param, value in chkpt_median_params.items():
        print(f"  {param}: {value:.6f}")
    
    # Calculate and print maximum log likelihood
    if 'log_likelihood' in samples_cumulative:
        log_lik_samples = samples_cumulative['log_likelihood'].flatten()
        max_log_lik = float(jnp.max(log_lik_samples))
        print(f"Maximum log likelihood: {max_log_lik:.6f}")
        
        # Also compute Bayesian Information Criterion (BIC) if you want
        n_params = len([p for p in samples_cumulative.keys() if p != 'log_likelihood'])
        n_data = len(SED_wavelengths_G102) + len(SED_wavelengths_G141)  # number of data points
        bic = -2 * max_log_lik + n_params * jnp.log(n_data)
        print(f"BIC: {bic:.6f}")

    with open(f'../../data/samples/samples_cumulative_{self.start_time}_checkpoint_{self.checkpoint:04d}.pkl', 'wb') as file:
        pickle.dump(dict(samples_cumulative), file)

class MCMCWithCheckpoints(MCMC):
    running_states = None
    checkpoint = 0
    start_time = None
    
    def run_checkpoints(self, rng_key, *args, extra_fields=(), n_checkpoints=10, 
                        progress_bar_warmup=True, progress_bar_samples=True, 
                        init_params=None, on_checkpoint=None, **kwargs):
        """
        Run the MCMC samplers and collect samples.

        :param random.PRNGKey rng_key: Random number generator key to be used for the sampling.
            For multi-chains, a batch of `num_chains` keys can be supplied. If `rng_key`
            does not have batch_size, it will be split in to a batch of `num_chains` keys.
        :param args: Arguments to be provided to the :meth:`numpyro.infer.mcmc.MCMCKernel.init` method.
            These are typically the arguments needed by the `model`.
        :param extra_fields: Extra fields (aside from `"z"`, `"diverging"`) from the
            state object (e.g. :data:`numpyro.infer.hmc.HMCState` for HMC) to be collected
            during the MCMC run. Note that subfields can be accessed using dots, e.g.
            `"adapt_state.step_size"` can be used to collect step sizes at each step. Exclude sample sites from
            collection with "~`sampler.sample_field`.`sample_site`". e.g. "~z.a" will prevent site "a" from
            being collected if you're using the NUTS sampler. To collect samples of a site "a" in the
            unconstrained space, we can specify the variable here, e.g. `extra_fields=("z.a",)`.
        :type extra_fields: tuple or list of str
        :param init_params: Initial parameters to begin sampling. The type must be consistent
            with the input type to `potential_fn` provided to the kernel. If the kernel is
            instantiated by a numpyro model, the initial parameters here correspond to latent
            values in unconstrained space.
        :param kwargs: Keyword arguments to be provided to the :meth:`numpyro.infer.mcmc.MCMCKernel.init`
            method. These are typically the keyword arguments needed by the `model`.

        .. note:: jax allows python code to continue even when the compiled code has not finished yet.
            This can cause troubles when trying to profile the code for speed.
            See https://jax.readthedocs.io/en/latest/async_dispatch.html and
            https://jax.readthedocs.io/en/latest/profiling.html for pointers on profiling jax programs.
        """
        self.start_time = datetime.now().strftime("%Y-%m-%d_%H-%M")
        num_warmup_total = int(self.num_warmup)
        num_samples_total = int(self.num_samples)
        
        check_point_indices = [
            jnp.arange(num_warmup_total), 
            *jnp.array_split(jnp.arange(num_samples_total), n_checkpoints)
        ]
        n_checkpoints = len(check_point_indices)
        rng_keys = random.split(rng_key, n_checkpoints)
        pbar = tqdm(enumerate(zip(rng_keys, check_point_indices)), total=n_checkpoints)
        for checkpoint, (rng_key, bounds) in pbar:
            if checkpoint == 0:
                self.progress_bar = progress_bar_warmup
                pbar.set_description('Run warmup')
                print_big_message("Begin warmup")
                self.warmup(rng_key, *args, extra_fields=extra_fields, init_params=init_params, **kwargs)
                print_big_message(f"Begin {num_samples_total} samples with {n_checkpoints} checkpoints")

            else:
                pbar.set_description(f'Run samples {bounds.min()} to {bounds.max()}')

                self.progress_bar = progress_bar_samples
                self.num_samples = bounds.size
                self.run(rng_key, *args, extra_fields=extra_fields, init_params=init_params, **kwargs)

                # add to running states:
                if self.running_states is None:
                    self.running_states = dict(self._states)
                else:
                    hstack_recursive(self.running_states, self._states)
                
                # ensure that calls to `self.get_samples` will build a new samples array
                # out of the running states:
                self._states_flat = None
                self._states = self.running_states

                if on_checkpoint is not None:
                    on_checkpoint(self, **kwargs)
                self.checkpoint += 1
        
        pbar.close()

        # reset to total number for arviz IO
        self.num_samples = num_samples_total

## Define Plotting Function

In [ ]:
def plot_sed_fit_results(mcmc, model_designation, n_temps):
    samples = mcmc.get_samples()
    result = arviz.from_numpyro(mcmc)
    
    log_likelihoods = samples['log_likelihood']
    # Find the index where log_likelihood is closest to the median
    median_log_likelihood = jnp.median(log_likelihoods)
    max_likelihood_idx = jnp.argmax(log_likelihoods)
    # max_likelihood_idx = jnp.argmin(jnp.abs(log_likelihoods - median_log_likelihood))
    max_likelihood_params = {key: samples[key][max_likelihood_idx] for key in samples.keys()}
    
    'Random indices for posterior samples'
    n_total_samples = len(samples['log_likelihood'])
    random_indices = np.random.choice(n_total_samples, size=min(200, n_total_samples), replace=False)
    
    'Extract temperatures'
    temps = [3891,3020,6163]
    print(f"\nTemperature components: {[f'{t:.0f}K' for t in temps]}")
    
    'Calculate filling factors for all samples'
    n_samples = n_total_samples
    all_filling_factors = np.zeros((n_samples, n_temps))
    for idx in range(n_samples):
        scales = [1.0] + [10**samples[f'log_ff{i}'][idx] for i in range(2, n_temps + 1)]
        total_scale = sum(scales)
        all_filling_factors[idx, :] = [scale / total_scale for scale in scales]
    
    'FF Statistics'
    ff_medians = np.median(all_filling_factors, axis=0)
    ff_percentiles = np.percentile(all_filling_factors, [16, 84], axis=0)
    ff_means = np.mean(all_filling_factors, axis=0)
    ff_stds = np.std(all_filling_factors, axis=0)
    print("\n" + "="*60)
    print("FILLING FACTORS FOR EACH COMPONENT (from full posterior)")
    print("="*60)
    for i in range(n_temps):
        print(f"Component {i+1} (T = {temps[i]:.0f}K):")
        print(f"  Median = {ff_medians[i]:.4f} ({ff_medians[i]*100:.2f}%)")
        print(f"  Mean ± Std = {ff_means[i]:.4f} ± {ff_stds[i]:.4f}")
        print(f"  16th-84th percentile = [{ff_percentiles[0,i]:.4f}, {ff_percentiles[1,i]:.4f}]")
        print(f"  Range = [{np.min(all_filling_factors[:,i]):.4f}, {np.max(all_filling_factors[:,i]):.4f}]")
    print(f"\nSum of median filling factors: {np.sum(ff_medians):.4f}")
    print("="*60 + "\n")
    
    #-------------------------------------------------------
    # Filling Factor plot
    fig_ff, axes_ff = plt.subplots(1, n_temps, figsize=(4*n_temps, 4), dpi=400)
    if n_temps == 1:
        axes_ff = [axes_ff]
    for i in range(n_temps):
        axes_ff[i].hist(all_filling_factors[:, i], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
        axes_ff[i].axvline(ff_medians[i], color='red', linestyle='--', label=f'Median: {ff_medians[i]:.3f}')
        axes_ff[i].axvline(ff_percentiles[0, i], color='orange', linestyle=':', label='16th/84th')
        axes_ff[i].axvline(ff_percentiles[1, i], color='orange', linestyle=':')
        axes_ff[i].set_xlabel(f'Filling Factor - Comp {i+1}')
        axes_ff[i].set_ylabel('Frequency')
        axes_ff[i].legend(fontsize=8)
        axes_ff[i].set_title(f'T{temps[i]:.0f}K')
    plt.suptitle('Filling Factor Distributions')
    plt.tight_layout()
    plt.savefig(f'../../figs/{model_designation}_filling_factor_distributions.png', dpi=200)
    #-------------------------------------------------------
    
    # Extract wavelength shift and stretch parameters
    # wl_shift_G102 = max_likelihood_params.get('wl_shift_G102', 0.0)
    # wl_shift_G141 = max_likelihood_params.get('wl_shift_G141', 0.0)
    # wl_stretch_G102 = max_likelihood_params.get('wl_stretch_G102', 1.0)
    # wl_stretch_G141 = max_likelihood_params.get('wl_stretch_G141', 1.0)
    sigma_conv = 1.0
    
    # Adjust the observed wavelengths for each grism separately
    # adjusted_wavelengths_G102 = (SED_wavelengths_G102 - jnp.median(SED_wavelengths_G102)) * wl_stretch_G102 + jnp.median(SED_wavelengths_G102) + wl_shift_G102
    # adjusted_wavelengths_G141 = (SED_wavelengths_G141 - jnp.median(SED_wavelengths_G141)) * wl_stretch_G141 + jnp.median(SED_wavelengths_G141) + wl_shift_G141
    # combined_SED_wavelengths = jnp.concatenate([adjusted_wavelengths_G102, adjusted_wavelengths_G141])
    
    'Prepare data'
    spec_G102 = jnp.array(calibrated_mean_spec_G102 + jnp.asarray(max_likelihood_params['offset_G102'], dtype=jnp.float64))
    spec_G141 = jnp.array(calibrated_mean_spec_G141)
    _spec_flux_jax = jnp.concatenate([
        spec_G102 + max_likelihood_params['slope_G102'] * (SED_wavelengths_G102 - SED_wavelengths_G102[0]),
        spec_G141 + max_likelihood_params['slope_G141'] * (SED_wavelengths_G141 - SED_wavelengths_G141[0])
    ])
    max_likelihood_spec_flux = _spec_flux_jax / jnp.mean(_spec_flux_jax)
    max_likelihood_spec_err = jnp.concatenate([
        jnp.array(oot_spec_err_G102 * jnp.asarray(10**max_likelihood_params['beta_G102'], dtype=jnp.float64)),
        jnp.array(oot_spec_err_G141 * jnp.asarray(10**max_likelihood_params['beta_G141'], dtype=jnp.float64))
    ])
    combined_SED_wavelengths = jnp.concatenate([SED_wavelengths_G102, SED_wavelengths_G141])
        
    'Get spectra and scales for max likelihood'
    spectra = [get_BTSettl_spectrum_jax(T=Temp, metallicity=0) for Temp in temps]
    scales = [1.0] + [jnp.asarray(10**max_likelihood_params[f'log_ff{i}'], dtype=jnp.float64) for i in range(2, n_temps + 1)]
    
    'Max likelihood filling factors'
    total_scale = sum(scales)
    ml_filling_factors = [scale / total_scale for scale in scales]
    print("\nMaximum Likelihood Filling Factors:")
    for i, ff in enumerate(ml_filling_factors, 1):
        print(f"  Component {i}: {ff:.4f} ({ff*100:.2f}%)")
    
    'Calculate combined flux from components'
    _max_likelihood_flux = sum(scales[i] * spectra[i][1] for i in range(n_temps))
    component_fluxes = []
    for i in range(n_temps):
        comp_flux = scales[i] * spectra[i][1]
        comp_flux_norm = comp_flux / jnp.mean(_max_likelihood_flux)
        comp_binned = shone.bin_spectrum(combined_SED_wavelengths, panchromatic_wavelengths, comp_flux_norm)
        comp_convolved = convolve_spectrum_jax(combined_SED_wavelengths, comp_binned, sigma=sigma_conv)
        component_fluxes.append(comp_convolved / jnp.mean(comp_convolved))
    
    'Max Likelihood Combined spectrum'
    # _model_flux = jnp.sum(jnp.array(component_fluxes), axis=0)
    _model_flux = _max_likelihood_flux / jnp.mean(_max_likelihood_flux)
    binned_model_flux = shone.bin_spectrum(combined_SED_wavelengths, panchromatic_wavelengths, _model_flux)
    convolved = convolve_spectrum_jax(combined_SED_wavelengths, binned_model_flux, sigma=sigma_conv)
    max_likelihood_model_flux = convolved / jnp.mean(convolved)

    'Compute model flux'
    def compute_model_flux(params):
        temps = [jnp.asarray(params[f'T{i}'], dtype=jnp.float64) for i in range(1, n_temps + 1)]
        spectra = [get_BTSettl_spectrum_jax(T=Temp, metallicity=0) for Temp in temps]
        
        scales = [1.0] + [jnp.asarray(10**params[f'log_ff{i}'], dtype=jnp.float64) for i in range(2, n_temps + 1)]
        
        _flux = scales[0] * spectra[0][1]
        for i in range(1, n_temps):
            _flux += scales[i] * spectra[i][1]
        _model_flux = _flux / jnp.mean(_flux)

        wl_shift_G102_rand = params.get('wl_shift_G102', 0.0)
        wl_shift_G141_rand = params.get('wl_shift_G141', 0.0)
        wl_stretch_G102_rand = params.get('wl_stretch_G102', 1.0)
        wl_stretch_G141_rand = params.get('wl_stretch_G141', 1.0)
        
        rand_adj_wl_G102 = (SED_wavelengths_G102 - jnp.median(SED_wavelengths_G102)) * wl_stretch_G102_rand + jnp.median(SED_wavelengths_G102) + wl_shift_G102_rand
        rand_adj_wl_G141 = (SED_wavelengths_G141 - jnp.median(SED_wavelengths_G141)) * wl_stretch_G141_rand + jnp.median(SED_wavelengths_G141) + wl_shift_G141_rand
        rand_combined_wl = jnp.concatenate([rand_adj_wl_G102, rand_adj_wl_G141])
        
        _binned_model_flux = shone.bin_spectrum(rand_combined_wl, panchromatic_wavelengths, _model_flux)
        binned_model_flux = convolve_spectrum_jax(rand_combined_wl, _binned_model_flux, sigma=sigma_conv)

        
        
        return rand_combined_wl, binned_model_flux / jnp.mean(binned_model_flux)
    
    #-------------------------------------------------------
    # Model + Data Plot
    fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True, gridspec_kw={'height_ratios': [3, 1]}, dpi=400)
    fig.suptitle(f'0.8-1.64 μm SED | {n_temps}T Model', fontsize=17)
    axs[1].set_title('Residuals', fontsize=14)
    # axs[1].set_xlim(0.805, 1.64)
    # axs[1].set_ylim(-4, 4)
    axs[1].set_xlabel(r'Wavelength ($\mu$m)', fontsize=14)
    axs[0].set_ylabel('Relative Flux', fontsize=14)
    axs[1].set_ylabel(r'$\sigma$', fontsize=14)
    'Plot random posterior samples'
    for idx in random_indices:
        random_params = {key: samples[key][idx] for key in samples.keys()}
        random_wavelengths, random_model_flux = compute_model_flux(random_params)
        axs[0].plot(random_wavelengths[2:-5], random_model_flux[2:-5], color='red', alpha=0.05, linewidth=0.5, zorder=50)
    'Plot individual components'
    colors = ['green', 'purple', 'cyan', 'orange'][:n_temps]
    for i in range(n_temps):
        label = f'T = {temps[i]:.0f} K | ff = {scales[i]:.2f} ± {ff_stds[i]:.2f}'
        axs[0].plot(combined_SED_wavelengths[2:-5], scales[i] * component_fluxes[i][2:-5],
                    color=colors[i], linewidth=2, linestyle='--', label=label, zorder=100, alpha=0.5)
    'Plot Data'
    axs[0].errorbar(combined_SED_wavelengths[2:-5], max_likelihood_spec_flux[2:-5], yerr=max_likelihood_spec_err[2:-5],
                    fmt='o', color='b', ms=1, alpha=0.7, zorder=150, label='Data')
    'Plot Max Likelihood Model'
    axs[0].plot(combined_SED_wavelengths[2:-5], max_likelihood_model_flux[2:-5], color='k', linewidth=1, zorder=200, label='Model')
    axs[0].legend(loc='upper right', fontsize=10)
    'Plot Residuals'
    axs[1].errorbar(combined_SED_wavelengths[2:-5], (max_likelihood_model_flux[2:-5] - max_likelihood_spec_flux[2:-5]) / max_likelihood_spec_err[2:-5],
                   color='k', yerr=[1]*len(combined_SED_wavelengths[2:-5]), zorder=150, ms=1, fmt='o', alpha=0.7)
    axs[1].axhline(0, color='k', zorder=-100)
    plt.tight_layout()
    plt.savefig(f'../../figs/{model_designation}_max_likelihood_result.png')
    #-------------------------------------------------------
    # Plot the wavelength shifts
    # plt.figure(figsize=(6,4),dpi=400)
    # fig.suptitle(f'Wavelength Shifts', fontsize=17)
    # plt.scatter(combined_SED_wavelengths,(random_wavelengths-combined_SED_wavelengths)/jnp.diff(combined_SED_wavelengths).mean(),alpha=0.4,s=1)
    # plt.xlabel(r'Wavelength ($\mu$m)', fontsize=14)
    # plt.ylabel('(New-original)/binsize', fontsize=14)
    
    # plt.tight_layout()
    # plt.savefig(f'../../figs/{model_designation}_wavelength_shifts.png')
    # #-------------------------------------------------------
    # # Plot the wavelength shifts
    # plt.figure(figsize=(6,4),dpi=400)
    # fig.suptitle(f'Wavelength Shifts', fontsize=17)
    # plt.plot(random_wavelengths[2:-5],max_likelihood_spec_flux[2:-5],alpha=0.4)
    # plt.plot(combined_SED_wavelengths[2:-5],max_likelihood_spec_flux[2:-5],alpha=0.4)
    # plt.xlabel(r'Wavelength ($\mu$m)', fontsize=14)
    # # plt.ylabel('(New-original)/binsize', fontsize=14)
    
    # plt.tight_layout()
    # plt.savefig(f'../../figs/{model_designation}_wavelength_shifts.png')
    #-------------------------------------------------------
    # Prepare corner plot data with derived ff parameters
    # This works for 1T, 2T, and 3T models
    
    # Define which variables to include in the corner plot
    # base_var_names = ['beta_G102', 'beta_G141', 'wl_shift_G102', 'wl_shift_G141',
    #                   'wl_stretch_G102', 'wl_stretch_G141', 'slope_G102', 'slope_G141', 'offset_G102']
    # Create dictionary for corner plot data
    corner_data = {}
    
    # Add base parameters
    # for var in base_var_names:
    #     if var in samples:
    #         corner_data[var] = np.array(samples[var])
    
    # Add temperature parameters
    for i in range(1, n_temps + 1):
        corner_data[f'T{i}'] = np.array(samples[f'T{i}'])
    
    # Add ff parameters based on n_temps
    if n_temps == 1:
        # 1T model: ff1 is always 1.0 (no variation, so we skip it in corner plot)
        # If you still want to show it, uncomment the next line
        # corner_data['ff1'] = np.ones(n_samples)
        pass  # No ff parameters needed for 1T model
    
    elif n_temps == 2:
        # 2T model: ff2 = 10**log_ff2, ff1 = 1.0 - ff2
        ff2 = all_filling_factors[:, 1]  # Component 2 from the earlier calculation
        ff1 = 1.0 - ff2                  # Component 1 derived
        
        corner_data['ff1'] = ff1
        corner_data['ff2'] = ff2
    
    elif n_temps == 3:
        # 3T model: ff2 = 10**log_ff2, ff3 = 10**log_ff3, ff1 = 1.0 - (ff2 + ff3)
        ff2 = all_filling_factors[:, 1]  # Component 2
        ff3 = all_filling_factors[:, 2]  # Component 3
        ff1 = 1.0 - (ff2 + ff3)          # Component 1 derived
        
        corner_data['ff1'] = ff1
        corner_data['ff2'] = ff2
        corner_data['ff3'] = ff3
    
    # Convert to array for corner plot
    var_names_list = list(corner_data.keys())
    samples_array = np.column_stack([corner_data[key] for key in var_names_list])
    
    # Create corner plot
    figure = corner.corner(
        samples_array,
        labels=var_names_list,
        show_titles=True,
        title_fmt='.3f',
        quantiles=[0.16, 0.5, 0.84]
    )
    
    plt.savefig(f'../../figs/{model_designation}_corner.png', dpi=400)
    plt.close()
    #-------------------------------------------------------

## Run 1T Model

In [ ]:
bt_lib = 'agss-logg4.5-met0.0' # GNS93, generic, CIFIST, AGSS2009
n_temps = 1
n_warmup = 5_00
n_samples = 2_000
n_chains = 8
n_check = 2

# Get the model function and callback
model_func, get_max_likelihood = numpyro_model(N_temps=n_temps, return_max_likelihood=True)

model_designation = f'jointvisit_{n_temps}T_{n_samples}_{n_chains}chains_SED_btsettl_{bt_lib}'

rng_key = PRNGKey(42)

sampler = NUTS(
    model_func,  # Use the returned function
    dense_mass=False
)

mcmc = MCMCWithCheckpoints(
    sampler, 
    num_warmup=n_warmup, 
    num_samples=n_samples,
    num_chains=n_chains
)

mcmc.run_checkpoints(rng_key, n_checkpoints=n_check, on_checkpoint=post_batch_viz_save)

mcmc.print_summary()

result = arviz.from_numpyro(mcmc)

'Save the result'
result.to_netcdf(f'../../data/samples/{model_designation}_{datetime.now().strftime("%Y-%m-%d")}')

# Extract maximum likelihood after sampling
if get_max_likelihood is not None:
    samples = mcmc.get_samples()
    max_log_lik = get_max_likelihood(samples)
    print(f"Maximum log likelihood: {max_log_lik:.6f}")

'Print the summary'
arviz.summary(result)

In [ ]:
'Examine the Leave-One-Out (LOO) summary'
loo = arviz.loo(result, pointwise=True)
loo

In [ ]:
# View the Pareto k values for every data point
print(loo.pareto_k)

bad_indices = np.where(loo.pareto_k.values > 0.7)[0]
print(f"Indices of 'very bad' points: {bad_indices}")

# Find the indices of points with a "very bad" diagnostic (k > 1)
very_bad_indices = np.where(loo.pareto_k.values > 1.0)[0]
print(f"Indices of 'very bad' points: {very_bad_indices}")

In [ ]:
plot_sed_fit_results(mcmc, model_designation, n_temps)

## Run 2T Model

In [ ]:
bt_lib = 'agss-logg4.5-met0.0' # GNS93, generic, CIFIST, AGSS2009
n_temps = 2
n_warmup = 5_00
n_samples = 2_000
n_chains = 8
n_check = 2

# Get the model function and callback
model_func, get_max_likelihood = numpyro_model(N_temps=n_temps, return_max_likelihood=True)

model_designation = f'jointvisit_{n_temps}T_{n_samples}_{n_chains}chains_SED_btsettl_{bt_lib}'

rng_key = PRNGKey(42)

sampler = NUTS(
    model_func,  # Use the returned function
    dense_mass=False
)

mcmc = MCMCWithCheckpoints(
    sampler, 
    num_warmup=n_warmup, 
    num_samples=n_samples,
    num_chains=n_chains
)

mcmc.run_checkpoints(rng_key, n_checkpoints=n_check, on_checkpoint=post_batch_viz_save)

mcmc.print_summary()

result = arviz.from_numpyro(mcmc)

In [ ]:
'Save the result'
result.to_netcdf(f'../../data/samples/{model_designation}_{datetime.now().strftime("%Y-%m-%d")}')

# Extract maximum likelihood after sampling
if get_max_likelihood is not None:
    samples = mcmc.get_samples()
    max_log_lik = get_max_likelihood(samples)
    print(f"Maximum log likelihood: {max_log_lik:.6f}")

'Print the summary'
arviz.summary(result)

In [ ]:
'Examine the Leave-One-Out (LOO) summary'
loo = arviz.loo(result, pointwise=True)
loo

In [ ]:
# View the Pareto k values for every data point
print(loo.pareto_k)

bad_indices = np.where(loo.pareto_k.values > 0.7)[0]
print(f"Indices of 'very bad' points: {bad_indices}")

# Find the indices of points with a "very bad" diagnostic (k > 1)
very_bad_indices = np.where(loo.pareto_k.values > 1.0)[0]
print(f"Indices of 'very bad' points: {very_bad_indices}")

In [ ]:
plot_sed_fit_results(mcmc, model_designation, n_temps)

## Run 3T Model

In [ ]:
bt_lib = 'agss-logg4.5-met0.0' # GNS93, generic, CIFIST, AGSS2009
n_temps = 3
n_warmup = 5_00
n_samples = 2_000
n_chains = 8
n_check = 2

# Get the model function and callback
model_func, get_max_likelihood = numpyro_model(N_temps=n_temps, return_max_likelihood=True)

model_designation = f'jointvisit_{n_temps}T_{n_samples}_{n_chains}chains_SED_btsettl_{bt_lib}'

rng_key = PRNGKey(42)

sampler = NUTS(
    model_func,  # Use the returned function
    dense_mass=False
)

mcmc = MCMCWithCheckpoints(
    sampler, 
    num_warmup=n_warmup, 
    num_samples=n_samples,
    num_chains=n_chains
)

mcmc.run_checkpoints(rng_key, n_checkpoints=n_check, on_checkpoint=post_batch_viz_save)

mcmc.print_summary()

# Extract maximum likelihood after sampling
if get_max_likelihood is not None:
    samples = mcmc.get_samples()
    max_log_lik = get_max_likelihood(samples)
    print(f"Maximum log likelihood: {max_log_lik:.6f}")

result = arviz.from_numpyro(mcmc)

'Save the result'
result.to_netcdf(f'../../data/samples/{model_designation}_{datetime.now().strftime("%Y-%m-%d")}')

'Print the summary'
arviz.summary(result)

In [ ]:
'Examine the Leave-One-Out (LOO) summary'
loo = arviz.loo(result, pointwise=True)
loo

In [ ]:
# View the Pareto k values for every data point
print(loo.pareto_k)

bad_indices = np.where(loo.pareto_k.values > 0.7)[0]
print(f"Indices of 'very bad' points: {bad_indices}")

# Find the indices of points with a "very bad" diagnostic (k > 1)
very_bad_indices = np.where(loo.pareto_k.values > 1.0)[0]
print(f"Indices of 'very bad' points: {very_bad_indices}")

In [ ]:
plot_sed_fit_results(mcmc, model_designation, n_temps)

# Compare multiple models of the same dataset

In [ ]:
# import arviz_plots as azp

# compare_dict = {}

# # BT-Settl generic models
# # compare_dict['BT_Settl_generic_1T'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_1T_2000_8chains_SED_btsettl_generic-logg4.5')
# # compare_dict['BT_Settl_generic_2T'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_2T_2000_8chains_SED_btsettl_generic-logg4.5')
# # compare_dict['BT_Settl_generic_3T'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_3T_2000_8chains_SED_btsettl_generic-logg4.5')

# # BT-Settl AGSS Models
# compare_dict['BT_Settl_agss_1T'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_1T_2000_8chains_SED_btsettl_agss-logg4.5')
# compare_dict['BT_Settl_agss_2T'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_2T_2000_8chains_SED_btsettl_agss-logg4.5')
# compare_dict['BT_Settl_agss_3T'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_3T_2000_8chains_SED_btsettl_agss-logg4.5')
# compare_dict['BT_Settl_agss_1T_met0'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_1T_1000_8chains_SED_btsettl_agss-logg4.5-met0.0_2026-05-13')
# compare_dict['BT_Settl_agss_2T_met0'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_2T_1000_8chains_SED_btsettl_agss-logg4.5-met0.0_2026-05-13')
# compare_dict['BT_Settl_agss_3T_met0'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_3T_1000_8chains_SED_btsettl_agss-logg4.5-met0.0_2026-05-13')

# # Sphinx Models
# # compare_dict['Sphinx1T'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_1T_2000_8chains_SED_sphinx_logg4.5')
# # compare_dict['Sphinx2T'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_2T_2000_8chains_SED_sphinx_logg4.5')
# # compare_dict['Sphinx3T'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_3T_2000_8chains_SED_sphinx_logg4.5')
# # compare_dict['Sphinx1T_met0'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_1T_2000_8chains_SED_sphinx_logg4.5_met0')
# # compare_dict['Sphinx2T_met0'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_2T_2000_8chains_SED_sphinx_logg4.5_met0')
# # compare_dict['Sphinx3T_met0'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_3T_2000_8chains_SED_sphinx_logg4.5_met0')

# arviz.compare(compare_dict,method='BB-pseudo-BMA')

In [ ]:
# compare_df = arviz.compare(compare_dict,method='BB-pseudo-BMA')
# azp.plot_compare(compare_df)

From https://num.pyro.ai/en/latest/tutorials/bad_posterior_geometry.html

In general it is difficult to assess whether the samples returned from HMC or NUTS represent accurate (approximate) samples from the posterior. Two general rules of thumb, however, are to look at the effective sample size (ESS) and r_hat diagnostics returned by mcmc.print_summary(). If we see values of r_hat in the range (1.0, 1.05) and effective sample sizes that are comparable to the total number of samples num_samples (assuming thinning=1) then we have good reason to believe that HMC is doing a good job. If, however, we see low effective sample sizes or large r_hats for some of the variables (e.g. r_hat = 1.15) then HMC is likely struggling with the posterior geometry. In the following we will use r_hat as our primary diagnostic metric.